# Análise Exploratória de Dados — NPS Preditivo
### Tech Challenge Fase 1 · FIAP AI Scientist

**Objetivo:** identificar quais fatores operacionais influenciam a satisfação do cliente, medida pelo NPS, e traduzir esses padrões em recomendações concretas para o negócio.

**Base de dados:** `desafio_nps_fase_1.csv` — 2.500 pedidos com dados de logística, atendimento ao cliente e indicadores internos.

---


## 1. Configuração do ambiente

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

# Estilo dos gráficos
sns.set_style("whitegrid")
plt.rcParams.update({
    "figure.dpi": 120,
    "font.size": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.figsize": (10, 4),
})

# Paleta de cores alinhada ao contexto NPS
CORES = {
    "detrator": "#E24B4A",
    "neutro":   "#EF9F27",
    "promotor": "#639922",
    "azul":     "#185FA5",
    "cinza":    "#888780",
}


## 2. Carga dos dados

A base está disponível no repositório do projeto. O caminho abaixo aponta para `data/desafio_nps_fase_1.csv` relativo à raiz do repositório — funciona localmente após clonar o repositório ou executar direto no GitHub Codespaces.


In [ ]:
# Caminho relativo à raiz do repositório
# Ajuste se estiver executando o notebook de uma subpasta diferente
DATA_PATH = Path("../data/desafio_nps_fase_1.csv")

# Fallback: tenta o mesmo diretório do notebook, caso esteja na raiz
if not DATA_PATH.exists():
    DATA_PATH = Path("data/desafio_nps_fase_1.csv")

df = pd.read_csv(DATA_PATH)
print(f"Base carregada: {df.shape[0]:,} registros · {df.shape[1]} variáveis")


## 3. Qualidade dos dados

Antes de qualquer análise, é preciso entender o que a base tem — e o que ela não tem.


In [ ]:
print("─" * 50)
print(f"Registros     : {df.shape[0]:,}")
print(f"Variáveis     : {df.shape[1]}")
print(f"Valores nulos : {df.isnull().sum().sum()}")
print(f"Duplicatas    : {df.duplicated().sum()}")
print("─" * 50)
print("\nTipos das variáveis:")
print(df.dtypes.to_string())


> **Observação:** a base não apresenta nulos nem duplicatas — qualidade excelente para análise direta, sem etapa de imputação.


## 4. Variável-alvo: `nps_score`

O NPS classifica clientes em três categorias com base na nota fornecida após a experiência de compra:

| Nota | Categoria |
|---|---|
| 0 a 6 | Detrator |
| 7 a 8 | Neutro |
| 9 a 10 | Promotor |

O NPS calculado é a diferença entre o percentual de promotores e detratores.


In [ ]:
# Classificação NPS
df["nps_categoria"] = pd.cut(
    df["nps_score"],
    bins=[-0.1, 6, 8, 10],
    labels=["Detrator", "Neutro", "Promotor"]
)

# Cálculo do NPS
pct_promotor = (df["nps_categoria"] == "Promotor").mean() * 100
pct_neutro   = (df["nps_categoria"] == "Neutro").mean() * 100
pct_detrator = (df["nps_categoria"] == "Detrator").mean() * 100
nps_calculado = pct_promotor - pct_detrator

print(f"Promotores : {pct_promotor:.1f}%  ({int(pct_promotor * len(df) / 100)} clientes)")
print(f"Neutros    : {pct_neutro:.1f}%  ({int(pct_neutro * len(df) / 100)} clientes)")
print(f"Detratores : {pct_detrator:.1f}%  ({int(pct_detrator * len(df) / 100)} clientes)")
print(f"\nNPS calculado: {nps_calculado:.0f} pontos")
print(f"Benchmark do setor (varejo online Brasil): +30 a +50 pontos")
print(f"→ A empresa está {abs(nps_calculado - 30):.0f} pontos abaixo do benchmark mínimo do setor.")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Histograma da distribuição das notas
axes[0].hist(df["nps_score"], bins=20, color=CORES["azul"], edgecolor="white", linewidth=0.5)
axes[0].set_xlabel("Nota NPS (0–10)")
axes[0].set_ylabel("Frequência")
axes[0].set_title("Distribuição das notas de NPS")

# Proporção por categoria
contagem = df["nps_categoria"].value_counts()
cores_pizza = [CORES["detrator"], CORES["neutro"], CORES["promotor"]]
axes[1].pie(
    contagem,
    labels=[f"{k}\n({v / len(df) * 100:.0f}%)" for k, v in contagem.items()],
    colors=cores_pizza,
    startangle=90,
    wedgeprops=dict(edgecolor="white", linewidth=1.5),
)
axes[1].set_title("Proporção por categoria NPS")

plt.tight_layout()
plt.savefig("../reports/fig_distribuicao_nps.png", bbox_inches="tight", dpi=150)
plt.show()


**Leitura para o negócio:** 74% dos clientes são detratores — isso não é um problema pontual, é um problema estrutural. O NPS de −66 está a mais de 90 pontos abaixo do benchmark mínimo do varejo online. Qualquer ação de melhoria precisa ser tratada como prioridade estratégica, não operacional.


## 5. O que mais impacta o NPS?

O primeiro passo é entender quais variáveis se movem junto com o NPS — para cima ou para baixo.


In [ ]:
# Correlação das variáveis numéricas com o nps_score
colunas_numericas = df.select_dtypes(include="number").columns.drop(
    ["nps_score", "customer_id", "order_id"]
)
correlacoes = df[colunas_numericas].corrwith(df["nps_score"]).sort_values()

# Visualização
fig, ax = plt.subplots(figsize=(9, 5))
cores_barras = [CORES["detrator"] if v < 0 else CORES["promotor"] for v in correlacoes.values]
correlacoes.plot(kind="barh", ax=ax, color=cores_barras, edgecolor="none")
ax.axvline(0, color=CORES["cinza"], linewidth=0.8, linestyle="--")
ax.set_xlabel("Correlação de Pearson com nps_score")
ax.set_title("O que derruba (vermelho) e o que eleva (verde) o NPS")
plt.tight_layout()
plt.savefig("../reports/fig_correlacoes_nps.png", bbox_inches="tight", dpi=150)
plt.show()

print("\nCorrelações com nps_score:")
print(correlacoes.round(3).to_string())


**Leitura para o negócio:**

Os três fatores que mais **derrubam** o NPS são:
1. `delivery_delay_days` (−0,60) — atraso na entrega é o principal vilão
2. `complaints_count` (−0,50) — volume de reclamações
3. `customer_service_contacts` (−0,35) — múltiplos contatos com o atendimento

Os dois fatores que mais **elevam** o NPS são:
1. `repeat_purchase_30d` (+0,57) — recompra em 30 dias é o sinal mais forte de satisfação real
2. `csat_internal_score` (+0,56) — score interno de satisfação

> **Atenção — risco de data leakage:** `csat_internal_score` tem correlação muito alta com o NPS e pode ter sido calculado a partir de informações posteriores à jornada. Antes de usá-lo como variável em um modelo preditivo, é necessário verificar sua origem temporal.


## 6. Ponto de ruptura: atraso na entrega

A variável com maior correlação negativa merece análise mais detalhada. A pergunta é: a partir de quantos dias de atraso a experiência do cliente começa a quebrar de forma significativa?


In [ ]:
# NPS médio por dias de atraso
nps_por_atraso = (
    df.groupby("delivery_delay_days")["nps_score"]
    .mean()
    .reset_index()
    .rename(columns={"nps_score": "nps_medio"})
)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(
    nps_por_atraso["delivery_delay_days"],
    nps_por_atraso["nps_medio"],
    marker="o", color=CORES["azul"], linewidth=2, markersize=8,
    zorder=3
)
# Linha de referência no ponto de ruptura
ax.axvline(x=2.5, color=CORES["detrator"], linestyle="--", linewidth=1.2,
           label="Ponto de ruptura (~3 dias)")
ax.fill_between(
    nps_por_atraso["delivery_delay_days"],
    nps_por_atraso["nps_medio"],
    alpha=0.08, color=CORES["azul"]
)
ax.set_xlabel("Dias de atraso na entrega")
ax.set_ylabel("NPS médio")
ax.set_title("NPS médio por dias de atraso na entrega")
ax.legend()
plt.tight_layout()
plt.savefig("../reports/fig_ponto_ruptura_atraso.png", bbox_inches="tight", dpi=150)
plt.show()

# Comparação direta
sem_atraso_alto = df[df["delivery_delay_days"] < 3]["nps_score"].mean()
com_atraso_alto = df[df["delivery_delay_days"] >= 3]["nps_score"].mean()
queda = (sem_atraso_alto - com_atraso_alto) / sem_atraso_alto * 100

print(f"NPS médio — atraso < 3 dias  : {sem_atraso_alto:.2f}")
print(f"NPS médio — atraso >= 3 dias : {com_atraso_alto:.2f}")
print(f"Queda ao cruzar o limiar     : {queda:.0f}%")


**Leitura para o negócio:** a partir de 3 dias de atraso, o NPS médio cai quase 50%. Esse é o limiar operacional mais claro identificado nos dados — garantir entrega em até 2 dias de atraso é o investimento com maior retorno direto em satisfação.


## 7. Perfil operacional: promotor vs detrator

Para além das correlações, é útil ver como cada grupo se comporta nas principais dimensões operacionais.


In [ ]:
metricas = [
    ("delivery_delay_days",        "Atraso médio (dias)"),
    ("complaints_count",           "Reclamações médias"),
    ("customer_service_contacts",  "Contatos de atendimento"),
    ("resolution_time_days",       "Tempo de resolução (dias)"),
]

fig, axes = plt.subplots(1, 4, figsize=(14, 4))

for ax, (col, label) in zip(axes, metricas):
    medias = df.groupby("nps_categoria")[col].mean()
    medias = medias.reindex(["Promotor", "Neutro", "Detrator"])
    cores_barras = [CORES["promotor"], CORES["neutro"], CORES["detrator"]]
    medias.plot(kind="bar", ax=ax, color=cores_barras, edgecolor="none", width=0.6)
    ax.set_title(label, fontsize=10, fontweight="bold")
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=20)
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.1f"))

plt.suptitle("Perfil operacional médio por categoria NPS", fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("../reports/fig_perfil_categorias.png", bbox_inches="tight", dpi=150)
plt.show()


**Leitura para o negócio:** o detrator não é apenas quem teve atraso — é quem teve atraso *e* reclamou *e* precisou entrar em contato com o atendimento mais de uma vez *e* esperou mais dias pela resolução. O problema se acumula em cascata: uma entrega atrasada gera reclamação, que gera contatos de atendimento, que se não resolvidos rápido viram detratores. Quebrar esse ciclo na origem — o atraso — é o caminho mais eficiente.


## 8. Recompra como sinal de satisfação real

A variável `repeat_purchase_30d` indica se o cliente voltou a comprar em até 30 dias. É o comportamento mais direto de fidelização — e tem correlação de +0,57 com o NPS.


In [ ]:
nps_recompra = df.groupby("repeat_purchase_30d")["nps_score"].mean()

fig, ax = plt.subplots(figsize=(5, 4))
nps_recompra.index = ["Sem recompra\n(30 dias)", "Com recompra\n(30 dias)"]
nps_recompra.plot(
    kind="bar", ax=ax,
    color=[CORES["detrator"], CORES["promotor"]],
    edgecolor="none", width=0.5
)
ax.set_title("NPS médio: recompra em 30 dias", fontweight="bold")
ax.set_ylabel("NPS médio")
ax.set_ylim(0, 10)
ax.tick_params(axis="x", rotation=0)

# Anotar valores nas barras
for bar in ax.patches:
    ax.annotate(
        f"{bar.get_height():.1f}",
        (bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1),
        ha="center", va="bottom", fontsize=11, fontweight="bold"
    )

plt.tight_layout()
plt.savefig("../reports/fig_nps_recompra.png", bbox_inches="tight", dpi=150)
plt.show()

print(f"Sem recompra : NPS médio {nps_recompra[0]:.2f}")
print(f"Com recompra : NPS médio {nps_recompra[1]:.2f}")
print(f"\nClientes que recompram em 30 dias: {df['repeat_purchase_30d'].mean()*100:.1f}% da base")


**Leitura para o negócio:** clientes que recompram em 30 dias têm NPS médio de 9,0 — são os promotores naturais da empresa. Representam apenas 9% da base, mas são o ativo mais valioso: custam menos para reter, compram mais e recomendam. Qualquer estratégia de CRM deve começar por identificar e proteger esse grupo.


## 9. NPS por região

O problema é sistêmico ou concentrado em alguma região específica?


In [ ]:
nps_regiao = df.groupby("customer_region")["nps_score"].mean().sort_values()
media_geral = df["nps_score"].mean()

fig, ax = plt.subplots(figsize=(7, 3))
nps_regiao.plot(kind="barh", ax=ax, color=CORES["azul"], edgecolor="none")
ax.axvline(media_geral, color=CORES["cinza"], linestyle="--", linewidth=1,
           label=f"Média geral ({media_geral:.1f})")
ax.set_xlabel("NPS médio")
ax.set_title("NPS médio por região")
ax.legend()
plt.tight_layout()
plt.savefig("../reports/fig_nps_regiao.png", bbox_inches="tight", dpi=150)
plt.show()

print("NPS médio por região:")
print(nps_regiao.round(2).to_string())
print(f"\nAmplitude entre regiões: {nps_regiao.max() - nps_regiao.min():.2f} pontos")


**Leitura para o negócio:** a variação entre regiões é mínima — menos de 0,3 pontos de amplitude. O problema não está concentrado em nenhuma região específica. Isso confirma que a causa é operacional e sistêmica, não geográfica.


## 10. Síntese dos principais achados

| # | Achado | Implicação para o negócio |
|---|---|---|
| 1 | NPS calculado de −66 (benchmark: +30 a +50) | O problema é estrutural — requer ação estratégica, não pontual |
| 2 | 74% dos clientes são detratores | Maioria da base está em risco de não voltar e de compartilhar experiências negativas |
| 3 | Atraso na entrega: maior correlação com NPS (−0,60) | Reduzir atraso é a alavanca com maior retorno direto em satisfação |
| 4 | Ponto de ruptura aos 3 dias de atraso | NPS cai ~50% ao cruzar esse limiar — é o SLA crítico a defender |
| 5 | Múltiplos contatos com atendimento amplificam a detração | O problema começa na entrega, mas escala no atendimento |
| 6 | Recompra em 30 dias: NPS médio de 9,0 | Clientes fiéis já existem na base — a estratégia é aumentar esse grupo |
| 7 | Variação regional mínima | Causa é operacional, não logística regional |

---

### Recomendações prioritárias

**Para logística:** estabelecer SLA interno de no máximo 2 dias de atraso como gatilho de alerta operacional. Pedidos que cruzam esse limiar devem entrar em protocolo de priorização imediata.

**Para atendimento:** monitorar em tempo real pedidos que geram segundo contato. O segundo contato é o sinal de que o problema não foi resolvido — e é onde a detração se consolida.

**Para CRM:** segmentar ativamente os 9% de clientes que recompraram em 30 dias. São os promotores naturais da base e o ponto de partida para qualquer estratégia de indicação e fidelização.
